In [6]:
import torch
print(torch.__version__)

2.8.0+cu126


In [7]:
torch.cuda.is_available()

True

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
w_out = torch.FloatTensor([-1,1,-0.5])
w_out.get_device()

-1

In [12]:
# Класс torch.FloatTensor такого параметра не имеет.
w_out_2 = torch.tensor([-1,1,-0.5], device=device)
w_out_2.get_device()

0

In [13]:
# Копирование тензора на GPU и обратно

w_out = w_out.to(device)
w_out

tensor([-1.0000,  1.0000, -0.5000], device='cuda:0')

In [18]:
w_out = w_out.cpu()
w_out.get_device()

-1

In [20]:
# Чтобы проводить операции над тензорами, они должны находиться на 1 устройстве.
w_out_2 = torch.tensor([-1,1,-0.5], device=device)
w_out_2.get_device()


0

In [21]:
c = w_out + w_out_2

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [22]:
# Важно - seed разный на всех устройствах

torch.cuda.manual_seed(123)
torch.cuda.manual_seed_all(123) # на всех GPU устройства

In [24]:
# Если мы хотим на всех GPU, при каждом новом запуске программы выдавал 1 и тот же рандом.
# Нужно прописать следующее:
torch.cuda.manual_seed(123)
torch.cuda.manual_seed_all(123) # на всех GPU устройства
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Простая NN на GPU

In [26]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def act(x):
    return 0 if x < 0.5 else 1


def go(house, rock, attr):
    X = torch.tensor([house, rock, attr], dtype=torch.float32, device=device)
    W_h = torch.tensor([[0.3, 0.3, 0],
                        [0.4, -0.5, 1]], device=device)

    W_out = torch.tensor([-1.0, 1.0], device=device)

    Z_h = torch.mv(W_h, X) # Сумма на входах нейронов скрытого слоя
    print(f'Значение сумм на нейронах скрытого слоя: {Z_h}')

    U_h = torch.tensor([act(x) for x in Z_h], dtype=torch.float32, device=device)
    print(f'Значение на вызодах нейронов скрытого слоя: {U_h}')

    Z_out = torch.dot(W_out, U_h)
    Y = act(Z_out)
    print(f'Выходное значение НС: {Y}')

    return Y

In [27]:
res = go(1,1,1)
if res == 1:
    print("Нравиться")
else:
    print("Нет")

Значение сумм на нейронах скрытого слоя: tensor([0.6000, 0.9000], device='cuda:0')
Значение на вызодах нейронов скрытого слоя: tensor([1., 1.], device='cuda:0')
Выходное значение НС: 0
Нет
